#  💻 **Descarga de Datos**

---

En este notebook se presenta el proceso de carga de los datos provenientes de la cooperativa. 

Se incluye la etapa de carga de datos de forma correcta, se realiza el renombramiento de las columnas para mejorar la claridad de los datos.

Adicionalmente, se contemplan posibles eliminaciones de columnas que no aportan valor al análisis.

In [1]:
from pathlib import Path
import pandas as pd

In [2]:
url_datos = Path("../data/raw/merged_df.csv")

datos_cooperativa = pd.read_csv(url_datos, low_memory=False)

datos_cooperativa.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12910 entries, 0 to 12909
Data columns (total 32 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Unnamed: 0          12910 non-null  int64  
 1   n.credito           12910 non-null  object 
 2   cartera             12910 non-null  object 
 3   plazo               12910 non-null  int64  
 4   vinculacion         12910 non-null  int64  
 5   v.cuota             12910 non-null  float64
 6   v.prestamo          12910 non-null  float64
 7   s.capital           12910 non-null  float64
 8   s.intereses         12910 non-null  int64  
 9   aportes             12910 non-null  int64  
 10  morosidad           12910 non-null  int64  
 11  garantias           12910 non-null  int64  
 12  valorgarantia       12910 non-null  int64  
 13  reestr              12910 non-null  int64  
 14  ctasahorros         12910 non-null  float64
 15  edad                12910 non-null  float64
 16  tipo

En este caso, el objetivo es predecir la variable `default`, la cual indica si un cliente ha incurrido en impago. Para evitar problemas de **data leakage**, se ha decidido eliminar la variable `morosidad`, ya que contiene información futura que no estaría disponible en el momento de realizar la predicción, lo que podría llevar a que el modelo obtenga resultados artificialmente altos al “hacer trampa”.


También decidimos no incluir las variables `Unnamed: 0` y `n.credito` debido a que no aportan información útil para el proceso de aprendizaje del modelo.

La columna `Unnamed: 0` corresponde únicamente a un índice generado automáticamente durante la exportación del dataset, por lo que no representa ninguna característica real de los clientes ni tiene relación con la variable objetivo (`default`). Es innecesario.

Por otro lado, `n.credito` es simplemente un identificador único asociado a cada crédito. Este tipo de variable funciona como una llave o código de registro y no contiene patrones predictivos sobre el comportamiento de pago de los clientes. Además, incluir identificadores únicos puede afectar negativamente el entrenamiento, ya que el modelo podría intentar memorizar registros en lugar de aprender relaciones generales entre las variables.


## **Renombramiento correcto de las columnas**

In [3]:
datos_cooperativa.columns

Index(['Unnamed: 0', 'n.credito', 'cartera', 'plazo', 'vinculacion', 'v.cuota',
       'v.prestamo', 's.capital', 's.intereses', 'aportes', 'morosidad',
       'garantias', 'valorgarantia', 'reestr', 'ctasahorros', 'edad',
       'tipoasociado', 'actividadeconomica', 'estado_cliente', 'departamento',
       'ciudad', 'sexo', 'curtotalingresos', 'curtotalegresos', 'intestrato',
       'actualización', 'default', 'puntaje_data', 'grupo_dptmto',
       'grupo_ciudad', 'grupo_edad', 'grupo_actividadeco'],
      dtype='object')

Se realizó una selección específica de las columnas a utilizar en el modelo, excluyendo la variable `morosidad` para evitar **data leakage**. 


In [4]:
columnas_a_usar = ['cartera', 
                   'plazo', 
                   'vinculacion', 
                   'v.cuota',
                   'v.prestamo', 
                   's.capital', 
                   's.intereses', 
                   'aportes', 
                   'garantias', 
                   'valorgarantia', 
                   'reestr', 
                   'ctasahorros', 
                   'edad',
                   'tipoasociado', 
                   'estado_cliente', 
                   'departamento',
                   'ciudad',
                   'sexo', 
                   'curtotalingresos', 
                   'curtotalegresos', 
                   'intestrato',
                   'actualización', 
                   'default', 
                   'puntaje_data', 
                   'grupo_dptmto',
                   'grupo_ciudad', 
                   'grupo_edad', 
                   'grupo_actividadeco']

In [5]:
datos_cooperativa_usar = pd.read_csv(url_datos, usecols=columnas_a_usar, low_memory=False)
datos_cooperativa_usar.head()

,cartera,plazo,vinculacion,v.cuota,v.prestamo,s.capital,s.intereses,aportes,garantias,valorgarantia,...,curtotalingresos,curtotalegresos,intestrato,actualización,default,puntaje_data,grupo_dptmto,grupo_ciudad,grupo_edad,grupo_actividadeco
0,consumo_sin_libranza,1827,8103,356849.0,15000000.0,12923538.0,123855,7741255,1,7741255,...,4597000.0,1500000.0,5.0,1,0,795.0,3,1,3,4
1,consumo_sin_libranza,1826,1434,2650409.0,100460000.0,31911361.0,263265,4601706,1,4601706,...,4597000.0,650000.0,5.0,1,0,836.0,3,5,3,4
2,consumo_sin_libranza,1826,573,791482.0,30000000.0,23844684.0,261477,530431,1,530431,...,4400000.0,2000000.0,4.0,0,1,709.0,3,7,2,4
3,consumo_sin_libranza,2922,1902,2860501.0,176000000.0,113842595.0,1008570,3023534,2,320385440,...,22020000.0,1500000.0,4.0,1,0,733.0,3,6,3,1
4,consumo_sin_libranza,2557,1902,987637.0,50300000.0,38521256.0,317167,1023082,2,320385440,...,22020000.0,1500000.0,4.0,1,0,695.0,3,6,3,1


Renombramos correctamente los nombres de algunas columnas

In [6]:
datos_cooperativa_final = datos_cooperativa_usar = datos_cooperativa_usar.rename(columns={"v.cuota": "valor_cuota",
                                                      "v.prestamo": "valor_prestamo",
                                                      "s.capital": "saldo_capital",
                                                      "s.intereses": "saldo_interes",
                                                      "actualización": "actualizacion",
                                                      })

datos_cooperativa_final.head()

,cartera,plazo,vinculacion,valor_cuota,valor_prestamo,saldo_capital,saldo_interes,aportes,garantias,valorgarantia,...,curtotalingresos,curtotalegresos,intestrato,actualizacion,default,puntaje_data,grupo_dptmto,grupo_ciudad,grupo_edad,grupo_actividadeco
0,consumo_sin_libranza,1827,8103,356849.0,15000000.0,12923538.0,123855,7741255,1,7741255,...,4597000.0,1500000.0,5.0,1,0,795.0,3,1,3,4
1,consumo_sin_libranza,1826,1434,2650409.0,100460000.0,31911361.0,263265,4601706,1,4601706,...,4597000.0,650000.0,5.0,1,0,836.0,3,5,3,4
2,consumo_sin_libranza,1826,573,791482.0,30000000.0,23844684.0,261477,530431,1,530431,...,4400000.0,2000000.0,4.0,0,1,709.0,3,7,2,4
3,consumo_sin_libranza,2922,1902,2860501.0,176000000.0,113842595.0,1008570,3023534,2,320385440,...,22020000.0,1500000.0,4.0,1,0,733.0,3,6,3,1
4,consumo_sin_libranza,2557,1902,987637.0,50300000.0,38521256.0,317167,1023082,2,320385440,...,22020000.0,1500000.0,4.0,1,0,695.0,3,6,3,1


In [7]:
datos_cooperativa_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12910 entries, 0 to 12909
Data columns (total 28 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   cartera             12910 non-null  object 
 1   plazo               12910 non-null  int64  
 2   vinculacion         12910 non-null  int64  
 3   valor_cuota         12910 non-null  float64
 4   valor_prestamo      12910 non-null  float64
 5   saldo_capital       12910 non-null  float64
 6   saldo_interes       12910 non-null  int64  
 7   aportes             12910 non-null  int64  
 8   garantias           12910 non-null  int64  
 9   valorgarantia       12910 non-null  int64  
 10  reestr              12910 non-null  int64  
 11  ctasahorros         12910 non-null  float64
 12  edad                12910 non-null  float64
 13  tipoasociado        12910 non-null  int64  
 14  estado_cliente      12910 non-null  int64  
 15  departamento        12909 non-null  object 
 16  ciud

In [8]:
print(Path.cwd())

c:\Users\belac\OneDrive\Documentos\Universidad\Ciencia de Datos y ML\Mlops\credit-risk-classification\notebooks


In [9]:
#Aqui subimos al proyecto raiz donde queremos guardar los datos procesados.
BASE_DIR = Path.cwd().parent

ruta_datos_procesados = BASE_DIR / "data" / "processed"

ruta_datos_procesados.mkdir(parents=True, exist_ok=True)

ruta_archivo = ruta_datos_procesados / "01_datos_crudos_cooperativa.parquet"

datos_cooperativa_final.to_parquet(ruta_archivo, index=False)